In [1]:
import numpy as np
import pandas as pd

# For reproducibility
np.random.seed(42)

In [2]:
df = pd.read_csv('/kaggle/input/datasets/ijk037/race-data/race_data.csv')

horses = [
    "Shadowfax", "Iron Duke", "Morningstar", "Red Tide",
    "Gallant Fox", "Blue Streak", "Copper Prince", "Last Chance"
]

In [3]:
params = {}

for horse in horses:
    mu = df[horse].mean()
    sigma = df[horse].std()
    
    # Avoid zero variance edge case
    sigma = max(sigma, 1e-6)
    
    params[horse] = (mu, sigma)

In [4]:
N_SIM = 100000

win_counts = {horse: 0 for horse in horses}

for _ in range(N_SIM):
    sampled_times = {}
    
    for horse in horses:
        mu, sigma = params[horse]
        sampled_times[horse] = np.random.normal(mu, sigma)
    
    winner = min(sampled_times, key=sampled_times.get)
    win_counts[winner] += 1

# Convert to probabilities
p = {horse: win_counts[horse] / N_SIM for horse in horses}

print("Estimated True Probabilities:")
for h in horses:
    print(f"{h}: {p[h]:.4f}")

Estimated True Probabilities:
Shadowfax: 0.3240
Iron Duke: 0.1382
Morningstar: 0.3207
Red Tide: 0.0685
Gallant Fox: 0.0561
Blue Streak: 0.0409
Copper Prince: 0.0284
Last Chance: 0.0232


In [5]:
f = {
    "Shadowfax": 0.08,
    "Iron Duke": 0.09,
    "Morningstar": 0.11,
    "Red Tide": 0.13,
    "Gallant Fox": 0.14,
    "Blue Streak": 0.15,
    "Copper Prince": 0.14,
    "Last Chance": 0.16
}

In [6]:
ev = {}

for horse in horses:
    ev[horse] = p[horse] * (0.85 / f[horse]) - 1

print("\nExpected Value (per £1):")
for h in horses:
    print(f"{h}: {ev[h]:.4f}")


Expected Value (per £1):
Shadowfax: 2.4424
Iron Duke: 0.3054
Morningstar: 1.4781
Red Tide: -0.5523
Gallant Fox: -0.6592
Blue Streak: -0.7682
Copper Prince: -0.8276
Last Chance: -0.8768


In [7]:
bankroll = 10000
fraction = 0.5  # fractional Kelly (safer)

bets = {}

for horse in horses:
    edge = p[horse] * (0.85 / f[horse]) - 1
    odds = (0.85 / f[horse]) - 1
    
    if edge > 0:
        kelly = edge / odds
        bets[horse] = bankroll * fraction * kelly
    else:
        bets[horse] = 0

# Normalize (just in case > bankroll)
total_bet = sum(bets.values())

if total_bet > bankroll:
    scale = bankroll / total_bet
    for horse in bets:
        bets[horse] *= scale

In [8]:
print("\nRecommended Stakes (£):")
for h in horses:
    print(f"{h}: £{bets[h]:.2f}")

print(f"\nTotal Bet: £{sum(bets.values()):.2f}")


Recommended Stakes (£):
Shadowfax: £1268.78
Iron Duke: £180.84
Morningstar: £1098.61
Red Tide: £0.00
Gallant Fox: £0.00
Blue Streak: £0.00
Copper Prince: £0.00
Last Chance: £0.00

Total Bet: £2548.23
